# Bank Marketing Campaign Analysis

## tl;dr

`bank.csv` is a 4,521-record subset of `bank-full.csv` (45,211 records), so the files must not be joined or appended. The overall subscription conversion rate is **11.7%**. Prior campaign success, prior contact, and cellular contact are strong historical signals. `duration` is a post-call field and is excluded from pre-call targeting to prevent leakage.

## Context & Methods

This notebook cleans and analyses the UCI Bank Marketing campaign data with Python's standard library. `bank-full.csv` is the canonical dataset; the smaller file is used only to verify the relationship between the supplied files.

### Key assumptions

- Each row is a campaign contact; the source has no unique customer ID or exact campaign date.
- Segment comparisons are historical associations, not causal effects.
- `duration` is known only after the call finishes and cannot be used to choose whom to call.

## Data

### 1. Set input and output paths

In [ ]:
from pathlib import Path

# Keep the notebook and the two input CSVs in the same folder before running.
BASE_DIR = Path.cwd()
FULL_FILE = BASE_DIR / 'bank-full.csv'
SAMPLE_FILE = BASE_DIR / 'bank.csv'
OUTPUT_FILE = BASE_DIR / 'bank_campaign_clean.csv'

assert FULL_FILE.exists(), f'Missing file: {FULL_FILE}'
assert SAMPLE_FILE.exists(), f'Missing file: {SAMPLE_FILE}'
print(f'Full file: {FULL_FILE.name}')
print(f'Smaller file: {SAMPLE_FILE.name}')

### 2. Load and standardize the source files

In [ ]:
import csv

SOURCE_COLUMNS = [
    'age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
    'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
    'previous', 'poutcome', 'y'
]
NUMBER_COLUMNS = {'age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'}

def read_bank_csv(file_path):
    with open(file_path, newline='', encoding='utf-8') as file:
        reader = csv.DictReader(file, delimiter=';')
        assert reader.fieldnames == SOURCE_COLUMNS, 'Unexpected Bank Marketing schema'
        rows = []
        for row in reader:
            clean = {column: row[column].strip().lower() for column in SOURCE_COLUMNS}
            for column in NUMBER_COLUMNS:
                clean[column] = int(clean[column])
            rows.append(clean)
    return rows

full_rows = read_bank_csv(FULL_FILE)
sample_rows = read_bank_csv(SAMPLE_FILE)
print(f'Loaded {len(full_rows):,} full-file rows and {len(sample_rows):,} smaller-file rows.')

### 3. Validate the relationship between files

In [ ]:
def row_key(row):
    return tuple(row[column] for column in SOURCE_COLUMNS)

full_keys = {row_key(row) for row in full_rows}
sample_keys = {row_key(row) for row in sample_rows}
is_subset = sample_keys.issubset(full_keys)
duplicate_rows = len(full_rows) - len(full_keys)
missing_source_values = sum(value == '' for row in full_rows for value in row.values())

print(f'Smaller file is a subset of full: {is_subset}')
print(f'Matching smaller-file records: {len(sample_keys):,} of {len(sample_rows):,}')
print(f'Exact duplicates in full file: {duplicate_rows:,}')
print(f'Missing source values: {missing_source_values:,}')
print('Decision: analyse bank-full.csv only; do not join or append bank.csv.')

### 4. Clean fields and address leakage

In [ ]:
def add_clean_fields(row):
    clean = row.copy()
    pdays = clean['pdays']
    clean['previously_contacted'] = 'yes' if pdays != -1 else 'no'
    clean['days_since_previous_contact'] = '' if pdays == -1 else pdays
    clean['age_band'] = ('under_30' if clean['age'] < 30 else '30_44' if clean['age'] < 45
                         else '45_59' if clean['age'] < 60 else '60_plus')
    clean['balance_band'] = ('negative' if clean['balance'] < 0 else 'zero' if clean['balance'] == 0
                            else '1_999' if clean['balance'] < 1000 else '1000_plus')
    # Post-call field: do not use for pre-call targeting or model training.
    clean['call_duration_seconds_post_call'] = clean.pop('duration')
    return clean

clean_rows = [add_clean_fields(row) for row in full_rows]
with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=clean_rows[0].keys())
    writer.writeheader()
    writer.writerows(clean_rows)

model_features = ['age', 'job', 'marital', 'education', 'balance', 'housing', 'loan',
                  'contact', 'day', 'month', 'campaign', 'previously_contacted',
                  'days_since_previous_contact', 'previous', 'poutcome']
print(f'Cleaned file written to: {OUTPUT_FILE.resolve()}')
print('Pre-call model features:', ', '.join(model_features))
print('Excluded leakage field: call_duration_seconds_post_call')

## Results

### 5. Calculate conversion rates by segment

In [ ]:
from collections import defaultdict

def conversion_by(rows, column):
    groups = defaultdict(lambda: {'contacts': 0, 'subscriptions': 0})
    for row in rows:
        group = str(row[column])
        groups[group]['contacts'] += 1
        groups[group]['subscriptions'] += row['y'] == 'yes'
    return sorted(
        [(group, values['contacts'], values['subscriptions'],
          100 * values['subscriptions'] / values['contacts']) for group, values in groups.items()],
        key=lambda item: item[3], reverse=True,
    )

overall_subscriptions = sum(row['y'] == 'yes' for row in clean_rows)
overall_rate = 100 * overall_subscriptions / len(clean_rows)
print(f'Overall: {overall_subscriptions:,} subscriptions / {len(clean_rows):,} contacts = {overall_rate:.1f}%')

for segment in ['poutcome', 'contact', 'previously_contacted']:
    print(f'\n{segment}')
    for group, contacts, subscriptions, rate in conversion_by(clean_rows, segment):
        print(f'  {group:14} {subscriptions:5}/{contacts:<5} {rate:5.1f}%')

### 6. Visualize the strongest historical signals

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

def plot_rates(rows, column, title):
    values = conversion_by(rows, column)
    labels = [item[0] for item in values]
    rates = [item[3] for item in values]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    bars = ax.bar(labels, rates, color='#2563eb')
    ax.set_title(title)
    ax.set_xlabel(column.replace('_', ' ').title())
    ax.set_ylabel('Subscription conversion rate (%)')
    ax.set_ylim(0, max(rates) * 1.2)
    for bar, rate in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f'{rate:.1f}%',
                ha='center', va='bottom')
    plt.show()

plot_rates(clean_rows, 'poutcome', 'Conversion by Previous Campaign Outcome')
plot_rates(clean_rows, 'contact', 'Conversion by Contact Channel')

### 7. Check outreach fatigue

In [ ]:
def attempt_band(attempts):
    if attempts == 1:
        return '1'
    if attempts <= 3:
        return '2-3'
    if attempts <= 5:
        return '4-5'
    return '6+'

attempt_rows = [dict(row, attempt_band=attempt_band(row['campaign'])) for row in clean_rows]
fatigue_results = conversion_by(attempt_rows, 'attempt_band')
fatigue_order = {'1': 1, '2-3': 2, '4-5': 3, '6+': 4}
fatigue_results.sort(key=lambda item: fatigue_order[item[0]])

for group, contacts, subscriptions, rate in fatigue_results:
    print(f'{group:4} attempts: {subscriptions:5}/{contacts:<5} = {rate:5.1f}%')

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot([item[0] for item in fatigue_results], [item[3] for item in fatigue_results],
        marker='o', linewidth=2, color='#dc2626')
ax.set_title('Conversion Declines with More Campaign Attempts')
ax.set_xlabel('Campaign attempt band')
ax.set_ylabel('Subscription conversion rate (%)')
plt.show()

### 8. Run an SQLite-compatible SQL example

The repository also includes ten reusable queries in `bank_campaign_queries.sql`.

In [ ]:
import sqlite3

connection = sqlite3.connect(':memory:')
columns = list(clean_rows[0].keys())
quoted_columns = ', '.join(f'"{column}"' for column in columns)
connection.execute('CREATE TABLE bank_campaign_clean (' + quoted_columns + ')')
connection.executemany(
    'INSERT INTO bank_campaign_clean VALUES (' + ','.join('?' for _ in columns) + ')',
    [[row[column] for column in columns] for row in clean_rows],
)

query = '''
SELECT previously_contacted, contact, COUNT(*) AS contacts,
       ROUND(100.0 * AVG(CASE WHEN y = 'yes' THEN 1.0 ELSE 0 END), 1) AS conversion_pct
FROM bank_campaign_clean
GROUP BY previously_contacted, contact
ORDER BY conversion_pct DESC
'''

for result in connection.execute(query):
    print(result)

## Takeaways

1. Use `bank-full.csv` as the only analysis table. The smaller file is already contained in it.
2. Prioritize prior successful outcomes for follow-up testing: this historical segment has the highest conversion rate.
3. Favor cellular contact and improve unknown contact details. Unknown channels have much lower conversion.
4. Start with one contact and limit repeat outreach. The rate declines as campaign attempts rise.
5. Do not include call duration in targeting models. Validate future models by campaign time and customer ID once those fields are available.

These patterns are associations in historical campaign data. Test policy changes with a controlled experiment before operational rollout.